In [3]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, TrainingArguments, Trainer
#from transformers import BitsAndBytesConfig
import spacy
import torch
from yahooquery import search
from spacy.language import Language
from spacy import displacy
import time
import glob
import re
import math
import statistics
import os
import json
import calendar
import holidays
from pathlib import Path
from datetime import date
from datetime import datetime
import pandas as pd
import numpy as np
import collections
import hashlib
from dateutil.parser import parse
import shutil
import ast
from io import StringIO
import requests
import glob
import os

In [4]:
alias_file = "../../Summary/OTHER/aliases.json"
if os.path.exists(alias_file):
    with open(alias_file, 'r') as f:
        alias = json.load(f)
print(alias)

{'MAUS': 'MONTHLY ACTIVE USERS', 'ARR': 'ANNUAL RECURRING REVENUE', 'ARPU': 'ACTIVE REVENUE PER USER', 'ANNUAL RECURRING REVENUE ARR': 'ANNUAL RECURRING REVENUE', 'ANNUALIZED RECURRING REVENUE': 'ANNUAL RECURRING REVENUE', 'ANNUALIZED RECURRING REVENUE ARR': 'ANNUAL RECURRING REVENUE', 'NET NEW ARR': 'NET NEW ANNUAL RECURRING REVENUE', 'NET DOLLAR EXPANSION CUSTOMERS WITH MORE THAN 10 EMPLOYEES': 'NET DOLLAR EXPANSION', 'CUSTOMERS CONTRIBUTING MORE THAN $100,000 REVENUE': 'LARGE PAID CUSTOMERS', 'CUSTOMERS CONTRIBUTING MORE THAN $100,000 TTM REVENUE': 'LARGE PAID CUSTOMERS', 'CUSTOMERS WITH MORE THAN $100000 OF ARR': 'LARGE PAID CUSTOMERS', 'CUSTOMERS CONTRIBUTING MORE THAN $100,000 IN REVENUE': 'LARGE PAID CUSTOMERS', 'CUSTOMERS WITH MORE THAN 10 EMPLOYEES': 'TOTAL NUMBER OF PAID CUSTOMERS', 'BUSINESSES': 'TOTAL NUMBER OF PAID CUSTOMERS', 'NET NEW SUBSCRIPTION CUSTOMERS': 'NEW PAID CUSTOMERS', 'SUBSCRIPTION CUSTOMERS': 'TOTAL NUMBER OF PAID CUSTOMERS', 'CUSTOMER COUNT': 'TOTAL NUMBER 

In [5]:
reverse_alias = dict()
for key in alias.keys():
    val = alias[key]
    if(val not in reverse_alias):
        reverse_alias[val] = list()
    reverse_alias[val].append(key)
print(reverse_alias)

{'MONTHLY ACTIVE USERS': ['MAUS', 'MAUS-GLOBAL', 'GLOBAL MONTHLY ACTIVE USERS MAUS', 'GLOBAL MONTHLY ACTIVE USERS'], 'ANNUAL RECURRING REVENUE': ['ARR', 'ANNUAL RECURRING REVENUE ARR', 'ANNUALIZED RECURRING REVENUE', 'ANNUALIZED RECURRING REVENUE ARR', 'ANNUAL RUN-RATE REVENUE ARR', 'ANNUALIZED EXIT MONTHLY RECURRING SUBSCRIPTIONS ARR', 'RINGCENTRAL TOTAL ARR', 'ENDING ARR'], 'ACTIVE REVENUE PER USER': ['ARPU', 'AVERAGE REVENUE PER CUSTOMER ARPU', 'AVERAGE REVENUE PER CUSTOMER', 'ARPU-GLOBAL'], 'NET NEW ANNUAL RECURRING REVENUE': ['NET NEW ARR'], 'NET DOLLAR EXPANSION': ['NET DOLLAR EXPANSION CUSTOMERS WITH MORE THAN 10 EMPLOYEES', 'NET DOLLAR EXPANSION CUSTOMERS WITH GREATER THAN 10 EMPLOYEES', 'REVENUE RETENTION', 'NET DOLLAR EXPANSION TOTAL NUMBER OF CUSTOMERS', 'RETENTION RATE', 'SUBSCRIPTION REVENUE RETENTION RATE', 'NET REVENUE RETENTION', 'NET DOLLAR - BASED RETENTION RATE', 'SUBSCRIPTION REVENUE NET DOLLAR EXPANSION', 'NET DOLLAR RETENTION NDR', 'NET DOLLAR RETENTION', 'NET RET

In [110]:
org_to_sym = dict()
maxEntCount = 12
naStr = "Not enough information is available."
ansTemplateStr = dict()
ansTemplateStr["1"] = "*METRIC of *ORG in quarter *QTR year *YEAR was *RESULT. " 
ansTemplateStr["2"] = "*ORG had *COUNT *METRIC in quarter *QTR year *YEAR, these were:\n*RESULT"

In [82]:
def getOrgData(org):
    orgDataPath = "../../Summary/orgData/"+org+".txt"
    file = Path(orgDataPath)
    if file.is_file():
        #print(True)
        with open(orgDataPath) as f:
            data = json.load(f)
        #print(data)
        return data
    return None

In [83]:
def getOrgAttr(orgData, attr):
    if not orgData:
        return None
    asplit = attr.split("|")
    parent = asplit[0]
    if parent in orgData and "SOURCE" in orgData[parent]:
        src = orgData[parent]["SOURCE"]
        if src == "YH" or (parent == "ORGPROFILE" and src == "AD"):
            p = orgData
            for i in range(0, len(asplit)):
                if asplit[i] not in p:
                    return None
                p = p[asplit[i]]
            #print(p)
            return(p)
    return None

In [84]:
def getPrevQtr(qstr):
    if not qstr:
        return None
    prvQtr = None
    qs = qstr.split("-")[0]
    year = qstr.split("-")[1]
    if(qs == "Q1"):
        year = (int(year) - 1)
        prvQtr = "Q4-"+str(year)
    elif(qs == "Q2"):
        prvQtr = "Q1-"+str(year)
    elif(qs == "Q3"):
        prvQtr = "Q2-"+str(year)
    elif(qs == "Q4"):
        prvQtr = "Q3-"+str(year)
    return(prvQtr)

In [85]:
def getEntAttr(entData, attr):
    if not entData:
        return None
    asplit = attr.split("|")
    
    p = entData
    for i in range(0, len(asplit)):
        if asplit[i] not in p:
            return None
        p = p[asplit[i]]
    #print(p)
    return(p)

In [86]:
def getAttr(allEntities, source, attrList):
    if source not in allEntities:
        return None
    p = allEntities[source]
    if not attrList:
        return p
    asplit = attrList.split("|")
    for i in range(0, len(asplit)):
        if asplit[i] not in p:
            return None
        p = p[asplit[i]]
        #print(p)
    return(p)
    #return None

In [87]:
def isMetricPresent(source, metric):
    if metric not in source:
        return False
    if metric in source and "CONFLICT" in source[metric] and source[metric]["CONFLICT"]:
        return False
    return True

In [88]:
def name_to_symbol(company_name):
    results = search(company_name)
    if(results and "quotes" in results):
        return results["quotes"][0]["symbol"]
    return None

In [89]:
def get_earning_details(csym):
    entPath = "../../Summary/entities/"+csym+"-ENTITIES.json"
    entFile = Path(entPath)
    entities = None
    with open(entPath, encoding="utf-8") as f:
        entity = json.load(f)
        entities = entity[csym]
    return(entities)

In [90]:
def getEntities(sym):
    entPath = "../../Summary/entities/"+sym+"-ENTITIES.json"
    entFile = Path(entPath)
    entities = None
    allEntities = None
    if entFile.is_file():
        with open(entPath, encoding="utf-8") as f:
            entity = json.load(f)
            entities = entity[sym]
            allEntities = dict()
            orgData = getOrgData(sym)
            allEntities["ORGDATA"] = orgData
            allEntities["ENTITIES"] = entities
            #allEntities["ENTITY"] = entity
    return allEntities

In [91]:
key_alias = dict()
key_alias["POSITIVE FACTS"] = "POSFACTS"
key_alias["NEGATIVE FACTS"] = "NEGFACTS"
key_alias["POSITIVE POINTS"] = "POSFACTS"
key_alias["NEGATIVE POINTS"] = "NEGFACTS"
key_alias["EPS YOY"] = "EPS-YOY"
print(key_alias)

{'POSITIVE FACTS': 'POSFACTS', 'NEGATIVE FACTS': 'NEGFACTS', 'POSITIVE POINTS': 'POSFACTS', 'NEGATIVE POINTS': 'NEGFACTS', 'EPS YOY': 'EPS-YOY'}


In [92]:
def buildArgsFromGenData(genData):
    arg_dict = dict()
    new_arg_dict = dict()
    
    arglist = genData["ARGS"].split("!!")
    for arg in arglist:
        argkey = arg.split(":")[0]
        argval = arg.split(":")[1]
        if(argkey not in arg_dict):
            arg_dict[argkey] = argval
    print(arg_dict)
     
    new_arg_dict["KEY"] = arg_dict["KEY"]
    new_arg_dict["RALIAS"] = dict() 
    if("KEY" in arg_dict and arg_dict["KEY"] != "NA"):
        if(arg_dict["KEY"] in key_alias):
            new_arg_dict["KEY"] = key_alias[arg_dict["KEY"]]
    new_arg_dict["RALIAS"][new_arg_dict["KEY"]] = arg_dict["KEY"]
    
    new_arg_dict["ORG"] = arg_dict["ORG"]
    new_arg_dict["SYM"] = arg_dict["ORG"]
    if("ORG" in arg_dict and arg_dict["ORG"] != "NA"):
        if(arg_dict["ORG"] not in org_to_sym):
            sym = name_to_symbol(arg_dict["ORG"])
            if(sym):
                org_to_sym[arg_dict["ORG"]] = sym
                new_arg_dict["SYM"] = sym
        else:
            new_arg_dict["SYM"] = org_to_sym[arg_dict["ORG"]]
    sym = new_arg_dict["SYM"]
            
    if("CALENDAR" in arg_dict and arg_dict["CALENDAR"] != "NA"):
        if(arg_dict["CALENDAR"] == "Q"):
            new_arg_dict["CALENDAR"] = "QUARTERLY"
        elif(arg_dict["CALENDAR"] == "Y"):
            new_arg_dict["CALENDAR"] = "YEARLY"
    
    if("QTR" in arg_dict and arg_dict["QTR"] != "NA" and "YEAR" in arg_dict and arg_dict["YEAR"] != "NA"):
        new_arg_dict["SEARCH"] = arg_dict["QTR"] + "-" + arg_dict["YEAR"]
    
    new_arg_dict["FIELDS"] = list()
    new_arg_dict["FIELDS"].append(new_arg_dict["KEY"])
    #if(arg_dict["CALENDAR"] == "QOQ" or arg_dict["CALENDAR"] == "YOY"):
    new_arg_dict["FIELDS"].append(new_arg_dict["KEY"]+"-QOQ")
    new_arg_dict["FIELDS"].append(new_arg_dict["KEY"]+"-YOY")
        
    allEntities = getEntities(sym)
    
    if not allEntities:
        return None
            
    print()
    print(new_arg_dict)
    new_arg_dict["ALLENTS"] = allEntities
    print()
    #print(allEntities["ENTITIES"])
    
    return new_arg_dict

In [93]:
def getTableData(args):
    gData = dict()
    gData["DATA"] = dict()
    gData["TEXT"] = dict()
    gData["INDEX"] = list()
    
    attr = args["ALLENTS"]["ENTITIES"]
    search = args["SEARCH"]
    fields = args["FIELDS"]
    calendar = args["CALENDAR"]
    if("COUNT" in args):
        count = args["COUNT"]
    else:
        count = maxEntCount
    
    found = list()
    
    gData["FIELDS"] = fields
    gData["CALENDAR"] = calendar
    gData["ORG"] = args["ORG"]
    gData["SYM"] = args["SYM"]
    gData["RALIAS"] = args["RALIAS"]
    cnt = 0
    
    for item in attr:
        #print(item)
        if "PUBLISH" in attr[item] and not attr[item]["PUBLISH"]:
            continue
        if("GUIDE" not in attr[item]):
            continue
        if(re.search(search, item)):
            #print(item, search)
            if(cnt > 0 and "d+" not in search): # Exact search so end it here
                break
            cnt = cnt + 1
            found = list()
            for metric in attr[item]:
                #print(metric)
                if(metric in fields):
                    found.append(metric)
                    #print(item,metric)
                    if(cnt == 1):
                        if("GUIDE" not in attr[item]):
                            continue
                        if("QOQ" in metric and "GUIDE-QOQ" in fields):
                            # Increase count to make room for guidance
                            count = count + 1
                            gindex = attr[item]["GUIDE"]+"-GUIDE"
                            gmetric = metric.replace("-QOQ","")
                            gmetric = gmetric+"-GUIDE-QOQ"
                            if(metric not in gData["DATA"]):
                                gData["DATA"][metric] = list()
                                gData["TEXT"][metric] = list()
                            #gindex = attr[item][gmetric]["QUARTER"]+"-GUIDE"
                            #gindex = "Next Quarter"
                            #if "YEAR" in calendar:
                            #    gindex = "Next Year"
                            if gindex not in gData["INDEX"]:
                                gData["INDEX"].append(gindex)
                            if gmetric in attr[item]:
                                if("NUMBER-PCT" in attr[item][gmetric]):
                                    gData["DATA"][metric].append(attr[item][gmetric]["NUMBER-PCT"])
                                    gData["TEXT"][metric].append(attr[item][gmetric]["RTEXT-PCT"])
                                else:
                                    gData["DATA"][metric].append(None)
                                    gData["TEXT"][metric].append("ND")
                            else:
                                gData["DATA"][metric].append(None)
                                gData["TEXT"][metric].append("ND")
                        elif("YOY" in metric and "GUIDE-YOY" in fields):
                            # Increase count to make room for guidance
                            count = count + 1
                            gindex = attr[item]["GUIDE"]+"-GUIDE"
                            gmetric = metric.replace("-YOY","")
                            gmetric = gmetric+"-GUIDE-YOY"
                            if(metric not in gData["DATA"]):
                                gData["DATA"][metric] = list()
                                gData["TEXT"][metric] = list()
                            #gindex = attr[item][gmetric]["QUARTER"]+"-GUIDE"
                            #gindex = "Next Quarter"
                            #if "YEAR" in calendar:
                            #    gindex = "Next Year"
                            if gindex not in gData["INDEX"]:
                                gData["INDEX"].append(gindex)
                            if gmetric in attr[item]:
                                if("NUMBER-PCT" in attr[item][gmetric]):
                                    gData["DATA"][metric].append(attr[item][gmetric]["NUMBER-PCT"])
                                    gData["TEXT"][metric].append(attr[item][gmetric]["RTEXT-PCT"])
                                else:
                                    gData["DATA"][metric].append(None)
                                    gData["TEXT"][metric].append("ND")
                            else:
                                gData["DATA"][metric].append(None)
                                gData["TEXT"][metric].append("ND")
                        elif("GUIDE-CSUS" in metric and "GUIDE-CSUS-ORIG" in fields):
                            # Increase count to make room for guidance
                            count = count + 1
                            gindex = attr[item]["GUIDE"]+"-GUIDE"
                            gmetric = metric+"-ORIG"
                            if(metric not in gData["DATA"]):
                                gData["DATA"][metric] = list()
                                gData["TEXT"][metric] = list()
                            #gindex = attr[item][gmetric]["QUARTER"]+"-GUIDE"
                            #gindex = "Next Quarter"
                            #if "YEAR" in calendar:
                            #    gindex = "Next Year"
                            if gindex not in gData["INDEX"]:
                                gData["INDEX"].append(gindex)
                            if gmetric in attr[item]:
                                if("NUMBER-MONEY" in attr[item][gmetric]):
                                    gData["DATA"][metric].append(attr[item][gmetric]["NUMBER-MONEY"])
                                    gData["TEXT"][metric].append(attr[item][gmetric]["RTEXT-MONEY"])
                                else:
                                    gData["DATA"][metric].append(None)
                                    gData["TEXT"][metric].append("ND")
                            else:
                                gData["DATA"][metric].append(None)
                                gData["TEXT"][metric].append("ND")
                        elif("GUIDANCE" in fields):
                            # Increase count to make room for guidance
                            count = count + 1
                            gindex = attr[item]["GUIDE"]+"-GUIDE"
                            gmetric = metric+"-GUIDE"
                            if(metric not in gData["DATA"]):
                                gData["DATA"][metric] = list()
                                gData["TEXT"][metric] = list()
                            #gindex = attr[item][gmetric]["QUARTER"]+"-GUIDE"
                            #gindex = "Next Quarter"
                            #if "YEAR" in calendar:
                            #    gindex = "Next Year"
                            if gindex not in gData["INDEX"]:
                                gData["INDEX"].append(gindex)
                            if gmetric in attr[item]:
                                if("NUMBER-MONEY" in attr[item][gmetric]):
                                    if("NUMBER-AVG-MONEY" in attr[item][gmetric]):
                                        gData["DATA"][metric].append(attr[item][gmetric]["NUMBER-AVG-MONEY"])
                                        gData["TEXT"][metric].append(attr[item][gmetric]["RTEXT-AVG-MONEY"])
                                    else:
                                        gData["DATA"][metric].append(attr[item][gmetric]["NUMBER-MONEY"])
                                        gData["TEXT"][metric].append(attr[item][gmetric]["RTEXT-MONEY"])
                                else:
                                    gData["DATA"][metric].append(None)
                                    gData["TEXT"][metric].append("ND")
                            else:
                                gData["DATA"][metric].append(None)
                                gData["TEXT"][metric].append("ND")
                        if("YEARLY" == calendar and "ALL" in search):
                            continue
                    if("NUMBER-MONEY" in attr[item][metric]):
                        #print(attr[item][metric]["RTEXT-MONEY"])
                        #Metric guide for latest quarter
                        if(metric not in gData["DATA"]):
                            gData["DATA"][metric] = list()
                            gData["TEXT"][metric] = list()
                        nitem = item
                        nitem = item.replace("ALL-","")
                        if nitem not in gData["INDEX"]:
                            gData["INDEX"].append(nitem)
                        if("NUMBER-AVG-MONEY" in attr[item][metric]):
                            gData["DATA"][metric].append(attr[item][metric]["NUMBER-AVG-MONEY"])
                            gData["TEXT"][metric].append(attr[item][metric]["RTEXT-AVG-MONEY"])
                        else:
                            gData["DATA"][metric].append(attr[item][metric]["NUMBER-MONEY"])
                            gData["TEXT"][metric].append(attr[item][metric]["RTEXT-MONEY"])
                    
                    elif("NUMBER-CD" in attr[item][metric]):
                        #print(attr[item][metric]["RTEXT-CD"])
                        if(metric not in gData["DATA"]):
                            gData["DATA"][metric] = list()
                            gData["TEXT"][metric] = list()
                        gData["DATA"][metric].append(attr[item][metric]["NUMBER-CD"])
                        gData["TEXT"][metric].append(attr[item][metric]["RTEXT-CD"])
                        nitem = item
                        nitem = item.replace("ALL-","")
                        if nitem not in gData["INDEX"]:
                            gData["INDEX"].append(nitem)
                    
                    elif("NUMBER-PCT" in attr[item][metric]):
                        if(metric not in gData["DATA"]):
                            gData["DATA"][metric] = list()
                            gData["TEXT"][metric] = list()
                        gData["DATA"][metric].append(attr[item][metric]["NUMBER-PCT"])
                        if("RTEXT-PCT" in attr[item][metric]):
                            gData["TEXT"][metric].append(attr[item][metric]["RTEXT-PCT"])
                        else:
                            gData["TEXT"][metric].append(attr[item][metric]["TEXT-PCT"])
                        nitem = item
                        nitem = item.replace("ALL-","")
                        if nitem not in gData["INDEX"]:
                            gData["INDEX"].append(nitem)
                    
                    elif(isinstance(attr[item][metric], list)):
                        if(metric not in gData["DATA"]):
                            gData["DATA"][metric] = list()
                            gData["TEXT"][metric] = list()
                        gData["DATA"][metric].append(attr[item][metric])
                        gData["TEXT"][metric].append(attr[item][metric])
                        nitem = item
                        nitem = item.replace("ALL-","")
                        if nitem not in gData["INDEX"]:
                            gData["INDEX"].append(nitem)
                    else:
                        if(metric not in gData["DATA"]):
                            gData["DATA"][metric] = list()
                            gData["TEXT"][metric] = list()
                        gData["DATA"][metric].append(None)
                        gData["TEXT"][metric].append("ND")
                        nitem = item
                        nitem = item.replace("ALL-","")
                        if nitem not in gData["INDEX"]:
                            gData["INDEX"].append(nitem)
            if(len(found)>0 and (len(found) < len(fields))):
                nf = set(found) ^ set(fields)
                nf = list(nf)
                #print(found, fields, nf)
                for metric in nf:
                    if(metric != "GUIDANCE" and metric != "GUIDE-YOY" and metric != "GUIDE-QOQ" and metric != "GUIDE-CSUS-ORIG"):
                        if(cnt == 1):
                            if("GUIDE-YOY" in nf and "YOY" in metric):
                                # Increase count to make room for guidance
                                count = count + 1
                                gindex = attr[item]["GUIDE"]+"-GUIDE"
                                if(metric not in gData["DATA"]):
                                    gData["DATA"][metric] = list()
                                    gData["TEXT"][metric] = list()
                                if gindex not in gData["INDEX"]:
                                    gData["INDEX"].append(gindex)
                                gData["DATA"][metric].append(None)
                                gData["TEXT"][metric].append("ND")
                            elif("GUIDE-QOQ" in nf and "QOQ" in metric):
                                # Increase count to make room for guidance
                                count = count + 1
                                gindex = attr[item]["GUIDE"]+"-GUIDE"
                                if(metric not in gData["DATA"]):
                                    gData["DATA"][metric] = list()
                                    gData["TEXT"][metric] = list()
                                if gindex not in gData["INDEX"]:
                                    gData["INDEX"].append(gindex)
                                gData["DATA"][metric].append(None)
                                gData["TEXT"][metric].append("ND")
                            elif("GUIDANCE" in nf):
                                # Increase count to make room for guidance
                                count = count + 1
                                gindex = attr[item]["GUIDE"]+"-GUIDE"
                                if(metric not in gData["DATA"]):
                                    gData["DATA"][metric] = list()
                                    gData["TEXT"][metric] = list()
                                if gindex not in gData["INDEX"]:
                                    gData["INDEX"].append(gindex)
                                gData["DATA"][metric].append(None)
                                gData["TEXT"][metric].append("ND")
                                
                            if(calendar == "YEARLY"):
                                #print("CONTINUING...", metric)
                                if(metric not in gData["DATA"]):
                                    gData["DATA"][metric] = list()
                                    gData["TEXT"][metric] = list()
                                gData["DATA"][metric].append(None)
                                gData["TEXT"][metric].append("ND")
                                continue
                        if(metric not in gData["DATA"]):
                            gData["DATA"][metric] = list()
                            gData["TEXT"][metric] = list()
                        gData["DATA"][metric].append(None)
                        gData["TEXT"][metric].append("ND")

            if(fields[0] in gData["DATA"] and len(gData["DATA"][fields[0]]) >= count):
                break
            
            if(len(found) == 0):
                for metric in fields:
                    if(metric != "GUIDANCE" and metric != "GUIDE-YOY" and metric != "GUIDE-QOQ" and metric != "GUIDE-CSUS-ORIG"):
                        if(metric not in gData["DATA"]):
                            gData["DATA"][metric] = list()
                            gData["TEXT"][metric] = list()
                        gData["DATA"][metric].append(None)
                        gData["TEXT"][metric].append("ND")
                        nitem = item
                        nitem = item.replace("ALL-","")
                        if nitem not in gData["INDEX"]:
                            gData["INDEX"].append(nitem)
            
    for metric in fields:
        if metric in gData["DATA"] and metric in gData["TEXT"]:
            gData["DATA"][metric].reverse()
            gData["TEXT"][metric].reverse()
    
    if "INDEX" in gData:
        gData["INDEX"].reverse()
            
    #print(gData)
    return gData

In [103]:
def showtxt(gData, tnum):
    ansAttr = dict()
    ansAttr["*ORG"] = gData["ORG"].title()
    ralias = gData["RALIAS"]
    ans = None
    for index, item in enumerate(gData["INDEX"]):
        qtr = item.split("-")[0]
        ansAttr["*QTR"] = qtr
        yr = item.split("-")[1]
        ansAttr["*YEAR"] = yr
        #print(qtr, yr)
        for metric in gData["TEXT"]:
            #print(metric)
            origMetricName = metric
            if(metric in ralias):
                origMetricName = ralias[metric]
            ansAttr["*METRIC"] = origMetricName.title()
            if(isinstance(gData["TEXT"][metric][index], list)):
                ansAttr["*COUNT"] = (len(gData["TEXT"][metric][index]))
                liststr = "\n".join(gData["TEXT"][metric][index])
                ansAttr["*RESULT"] = liststr
                #print("{} of {} in quarter {} year {} are:\n\n{}".format(origMetricName, org, qtr, yr, liststr))
                if not ans:
                    ans = ansTemplateStr[str(tnum)]
                else:
                    ans = ans + ansTemplateStr[str(tnum)]
                for attr in ansAttr:
                    if(attr in ans):
                        ans = ans.replace(attr, str(ansAttr[attr]))
            else:
                if(gData["TEXT"][metric][index] != "ND"):
                    val = gData["TEXT"][metric][index]
                    if(val.startswith("(")):
                        chg = "declined"
                    else:
                        chg = "grew"
                    if("YOY" in metric):
                        nmetric = metric.replace("-YOY","")
                        origMetricName = nmetric
                        if(nmetric in ralias):
                            origMetricName = ralias[nmetric]
                        origMetricName = origMetricName.title()
                        ans = ans + origMetricName + " " + chg + " " + val + " year over year. "
                    elif("QOQ" in metric):
                        nmetric = metric.replace("-QOQ","")
                        origMetricName = nmetric
                        if(nmetric in ralias):
                            origMetricName = ralias[nmetric]
                        origMetricName = origMetricName.title()
                        ans = ans + origMetricName + " " + chg + " " + val + " quarter over quarter. "
                    else:
                        origMetricName = metric
                        if(metric in ralias):
                            origMetricName = ralias[metric]
                        ansAttr["*METRIC"] = origMetricName.title()
                        if not ans:
                            ans = ansTemplateStr[str(tnum)]
                        else:
                            ans = ans + ansTemplateStr[str(tnum)]
                        ansAttr["*RESULT"] = val
                        for attr in ansAttr:
                            if(attr in ans):
                                ans = ans.replace(attr, str(ansAttr[attr]))
    if ans:
        print(ans)
    else:
        print(naStr)

In [111]:
def getAns(genData, gData):
    ans = genData["ANS"]
    asplit = ans.split("!!")
    away = asplit[0].split(":")[1]
    tnum = asplit[1].split(":")[1]
    if(away == "SHOWTXT"):
        showtxt(gData, tnum)

In [112]:
qa = [
        "List all positive facts of Appian from Quarter Q2 Year 2025.",
        "What was the revenue of Nvidia for Q2 2026?",
        "What was the YoY EPS growth of Cloudflare in quarter Q1 2024?",
        "List all companies those beat revenue estimates in current quarter in table format."
        "Show all operational metrics Qoq trend of Gitlab in table format."
        "What was the last reported quarter of Zoom?"
        "Compare year over year quarterly revenue growth between Palantir and UIPath."
        #"Show all positive and negative facts count of Trade Desk for last 3 years in chart"
]
print(qa)

qargs = dict()
qargs["ARGS"] = list()
qargs["INTENT"] = list()
qargs["ANS"] = list()
qargs["ARGS"].append("KEY:POSITIVE FACTS!!ORG:APPIAN!!QTR:Q2!!YEAR:2025!!CALENDAR:Q!!FILTER:NA!!SECTION:REGULAR!!HOW:EXACT!!SOURCE:ENTITY")
qargs["ANS"].append("FN:SHOWTXT!!T:2")
qargs["INTENT"].append("INFO")
qargs["ARGS"].append("KEY:REVENUE!!ORG:NVIDIA!!QTR:Q2!!YEAR:2026!!CALENDAR:Q!!FILTER:NA!!SECTION:REGULAR!!HOW:EXACT!!SOURCE:ENTITY")
qargs["ANS"].append("FN:SHOWTXT!!T:1")
qargs["INTENT"].append("INFO")
qargs["ARGS"].append("KEY:EPS-YOY!!ORG:CLOUDFLARE!!QTR:Q1!!YEAR:2024!!CALENDAR:Q!!FILTER:NA!!SECTION:REGULAR!!HOW:EXACT!!SOURCE:ENTITY")
qargs["ANS"].append("FN:SHOWTXT!!T:1")
qargs["INTENT"].append("INFO")
qargs["ARGS"].append("KEY:REVENUE-RESULT!!ORG:NA!!QTR:CURRENT!!YEAR:CURRENT!!CALENDAR:Q!!FILTER:VAL==TRUE!!SECTION:REGULAR!!HOW:EXACT!!SOURCE:ENTITY")
qargs["ANS"].append("COLS:LATEST QTR,BEAT?!!FN:SHOWTBL")
qargs["INTENT"].append("GROUP INFO")
qargs["ARGS"].append("KEY:NA!!ORG:GITLAB!!QTR:NA!!YEAR:NA!!CALENDAR:QOQ!!FILTER:KEY==OPERATIONAL!!SECTION:REGULAR!!HOW:FILTER!!SOURCE:ENTITY")
qargs["ANS"].append("COLS:QTR!!FN:SHOWTBL")
qargs["INTENT"].append("INFO")
qargs["ARGS"].append("KEY:NA!!ORG:ZOOM!!QTR:LATEST!!YEAR:LATEST!!CALENDAR:Q!!FILTER:NA!!SECTION:REGULAR!!HOW:EXACT!!SOURCE:ENTITY")
qargs["ANS"].append("TXT:Last reported quarter of *ORG was *QTR *YEAR.")
qargs["INTENT"].append("INFO")
qargs["ARGS"].append("KEY:REVENUE!!ORG:PALANTIR AND UIPATH!!QTR:NA!!YEAR:NA!!CALENDAR:YOY!!FILTER:NA!!SECTION:REGULAR!!HOW:EXACT!!SOURCE:ENTITY")
qargs["ANS"].append("COLS:QTR!!FN:SHOWTBL")
qargs["INTENT"].append("COMPARE")
#qargs["ARGS"].append("KEY:POSTIVE FACTS AND NEGATIVE FACTS!!ORG:Trade Desk!!QTR:NA!!YEAR:LAST 3!!CALENDAR:Q!!FILTER:NA!!SECTION:REGULAR!!HOW:COUNT!!SOURCE:ENTITY")
#qargs["ANS"].append("FN:SHOWCHT")
#qargs["INTENT"].append("INFO")
print(json.dumps(qargs))

['List all positive facts of Appian from Quarter Q2 Year 2025.', 'What was the revenue of Nvidia for Q2 2026?', 'What was the YoY EPS growth of Cloudflare in quarter Q1 2024?', 'List all companies those beat revenue estimates in current quarter in table format.Show all operational metrics Qoq trend of Gitlab in table format.What was the last reported quarter of Zoom?Compare year over year quarterly revenue growth between Palantir and UIPath.']
{"ARGS": ["KEY:POSITIVE FACTS!!ORG:APPIAN!!QTR:Q2!!YEAR:2025!!CALENDAR:Q!!FILTER:NA!!SECTION:REGULAR!!HOW:EXACT!!SOURCE:ENTITY", "KEY:REVENUE!!ORG:NVIDIA!!QTR:Q2!!YEAR:2026!!CALENDAR:Q!!FILTER:NA!!SECTION:REGULAR!!HOW:EXACT!!SOURCE:ENTITY", "KEY:EPS-YOY!!ORG:CLOUDFLARE!!QTR:Q1!!YEAR:2024!!CALENDAR:Q!!FILTER:NA!!SECTION:REGULAR!!HOW:EXACT!!SOURCE:ENTITY", "KEY:REVENUE-RESULT!!ORG:NA!!QTR:CURRENT!!YEAR:CURRENT!!CALENDAR:Q!!FILTER:VAL==TRUE!!SECTION:REGULAR!!HOW:EXACT!!SOURCE:ENTITY", "KEY:NA!!ORG:GITLAB!!QTR:NA!!YEAR:NA!!CALENDAR:QOQ!!FILTER:KEY==O

In [113]:
qcnt = 0
for q,intent,args,ans in zip(qa, qargs["INTENT"],qargs["ARGS"],qargs["ANS"]):
    #print(q)
    #print(intent) 
    #print(args) 
    #print(ans)
    #print()
    
    genData = dict()
    genData["INTENT"] = intent
    genData["ARGS"] = args
    genData["QUESTION"] = q
    genData["ANS"] = ans
    print(genData)
    print()
    
    new_args = buildArgsFromGenData(genData)
    #print(org_to_sym)
    
    gData = getTableData(new_args)
    #print(gData)
    
    print(q)
    print()
    getAns(genData, gData)
    print()
    
    qcnt = qcnt + 1
    if(qcnt >= 2):
        break

{'INTENT': 'INFO', 'ARGS': 'KEY:POSITIVE FACTS!!ORG:APPIAN!!QTR:Q2!!YEAR:2025!!CALENDAR:Q!!FILTER:NA!!SECTION:REGULAR!!HOW:EXACT!!SOURCE:ENTITY', 'QUESTION': 'List all positive facts of Appian from Quarter Q2 Year 2025.', 'ANS': 'FN:SHOWTXT!!T:2'}

{'KEY': 'POSITIVE FACTS', 'ORG': 'APPIAN', 'QTR': 'Q2', 'YEAR': '2025', 'CALENDAR': 'Q', 'FILTER': 'NA', 'SECTION': 'REGULAR', 'HOW': 'EXACT', 'SOURCE': 'ENTITY'}

{'KEY': 'POSFACTS', 'RALIAS': {'POSFACTS': 'POSITIVE FACTS'}, 'ORG': 'APPIAN', 'SYM': 'APPN', 'CALENDAR': 'QUARTERLY', 'SEARCH': 'Q2-2025', 'FIELDS': ['POSFACTS', 'POSFACTS-QOQ', 'POSFACTS-YOY']}

List all positive facts of Appian from Quarter Q2 Year 2025.

Appian had 38 Positive Facts in quarter Q2 year 2025, these were:
GAAP-EPS GREW 105.26% QUARTER OVER QUARTER IN Q2 2025
GAAP-EPS GREW 100.17% YEAR OVER YEAR IN Q2 2025
GAAP GROSS PROFIT GREW 18.39% YEAR OVER YEAR IN Q2 2025
SUBSCRIPTION REVENUE-CLOUD GREW 7.11% QUARTER OVER QUARTER IN Q2 2025
SUBSCRIPTION REVENUE-CLOUD GREW 20